# Transform Data

Goal of this notebook is to convert all race data into a flat-file of the following schema:

- RaceId 
- DriverId
- TeamId
- Position (Starting)
- Starting Tyre Compound
- Starting Tyre freshness (boolean)  

Pit stop 1   
- A column to measure time taken to first pit stop  
- Lap at which first pit stop occurred
- Time taken for first pit stop
- Compound after first pit stop
- Freshness of tyre after first pit stop  

Pit stop 2  
- A column to measure time taken to second pit stop
- Lap at which second pit stop occurred
- Time taken for second pit stop
- Compound after second pit stop
- Freshness of tyre after second pit stop 

Pit stop three
- A column to measure time taken to third pit stop (if it exists)
- Lap at which third pit stop occurred (if it exists)
- Time taken for third pit stop (if it exists)
- Compound after third pit stop (if it exists)
- Freshness of tyre after third pit stop (if it exists)

- Number of laps to change tires after rain begins
- Number of laps to change tires after rain ends
- Rainfall present on track
- Maximum duration of rainfall (if present)
- Maximum difference in weather temperature
- Gradient of weather temperature
- Safety car deployed (multiple columns storing the lap at which it occurred, maybe go up to 72)
- Pit stops during a safety car
- Matrix of [Angle, Distance] for given track

Target
- GridPosition (Finishing, this will be our target)

In [2]:
import fastf1
from pathlib import Path
import pandas as pd
from IPython.display import display

DATA_PATH = Path.cwd().parent / "data"
fastf1.Cache.enable_cache(DATA_PATH)

In [3]:
race = fastf1.get_session(2021, "Monaco", "R")
race.load()

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 33 completed the race distance 00:00.058000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['33', '55', '4', '11', '5', '10', '44', '18', '31', '99', '7', '3',

In [4]:
race.results['Position'] # starting

33     1.0
55     2.0
4      3.0
11     4.0
5      5.0
10     6.0
44     7.0
18     8.0
31     9.0
99    10.0
7     11.0
3     12.0
14    13.0
63    14.0
6     15.0
22    16.0
9     17.0
47    18.0
77    19.0
16    20.0
Name: Position, dtype: float64

In [5]:
race.results['GridPosition'] # finishing

33     2.0
55     4.0
4      5.0
11     9.0
5      8.0
10     6.0
44     7.0
18    13.0
31    11.0
99    10.0
7     14.0
3     12.0
14    17.0
63    15.0
6     18.0
22    16.0
9     19.0
47    20.0
77     3.0
16     1.0
Name: GridPosition, dtype: float64

In [6]:
race.laps.columns

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')

In [7]:
race.laps[['Driver', 'DriverNumber', 'LapNumber', 'PitInTime', 'FreshTyre', 'TyreLife', 'Compound', 'TrackStatus']]

,Driver,DriverNumber,LapNumber,PitInTime,FreshTyre,TyreLife,Compound,TrackStatus
0,VER,33,1.0,NaT,False,6.0,SOFT,1
1,VER,33,2.0,NaT,False,7.0,SOFT,1
2,VER,33,3.0,NaT,False,8.0,SOFT,1
3,VER,33,4.0,NaT,False,9.0,SOFT,1
4,VER,33,5.0,NaT,False,10.0,SOFT,1
...,...,...,...,...,...,...,...,...
1415,BOT,77,27.0,NaT,False,32.0,SOFT,1
1416,BOT,77,28.0,NaT,False,33.0,SOFT,1
1417,BOT,77,29.0,NaT,False,34.0,SOFT,1
1418,BOT,77,30.0,0 days 01:11:12.283000,False,35.0,SOFT,1


In [8]:
race.weather_data

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,0 days 00:00:25.218000,20.7,61.3,1015.0,False,36.9,329,0.1
1,0 days 00:01:25.220000,20.7,61.4,1015.0,False,36.8,0,0.1
2,0 days 00:02:25.225000,20.6,62.3,1015.0,False,36.8,307,0.1
3,0 days 00:03:25.234000,20.5,63.2,1015.1,False,36.1,258,0.1
4,0 days 00:04:25.234000,20.5,63.3,1015.0,False,36.4,285,0.2
...,...,...,...,...,...,...,...,...
130,0 days 02:10:25.668000,20.9,58.9,1015.8,False,33.4,161,0.4
131,0 days 02:11:25.671000,20.9,59.0,1015.8,False,34.0,170,0.5
132,0 days 02:12:25.670000,20.9,59.5,1015.9,False,34.4,157,0.6
133,0 days 02:13:25.683000,21.0,59.2,1015.8,False,34.4,161,0.2


In [9]:
race.get_circuit_info().corners

,X,Y,Number,Letter,Angle,Distance
0,-7381.106934,-4719.949219,1,,140.384631,198.734262
1,-3634.549072,-3769.346436,2,,103.015117,599.115705
2,-2275.649414,-3071.900879,3,,168.949842,754.411687
3,-2745.113281,-1956.109497,4,,-174.396257,883.348274
4,-1422.224854,-197.893692,5,,76.191057,1105.573257
5,-797.246765,-1127.089111,6,,-85.862753,1228.559954
6,-1016.884705,-477.596558,7,,-14.461416,1311.061635
7,-271.229309,-251.078979,8,,44.799495,1391.498401
8,-1126.092163,-3321.891602,9,,125.099753,1727.756519
9,-4159.989258,-4441.595215,10,,144.845611,2056.174720


In [15]:
start_end_positions = pd.concat([race.results['Position'],race.results['GridPosition']], axis=1).rename(columns={'Position': 'starting_position', 'GridPosition': 'target'})
start_end_positions


,starting_position,target
33,1.0,2.0
55,2.0,4.0
4,3.0,5.0
11,4.0,9.0
5,5.0,8.0
10,6.0,6.0
44,7.0,7.0
18,8.0,13.0
31,9.0,11.0
99,10.0,10.0


In [31]:
laps = race.laps[['Driver', 'DriverNumber', 'LapNumber', 'PitInTime', 'PitOutTime', 'FreshTyre', 'TyreLife', 'Compound', 'TrackStatus']]

In [62]:
import numpy as np
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)


(laps := race.laps
    # calcualte pit stop duration
    .assign(pit_duration=lambda x: (x.PitOutTime.shift(-1) - x.PitInTime).dt.total_seconds())
    # the lap at which the pit stop was made
    .assign(pit_stop=lambda x: np.where(x.pit_duration > 0, x.LapNumber.astype(int), None))
    # the strategy after the pit stop
    .assign(strategy_after_pit=lambda x: np.where(x.shift(-1).Compound.isin(['SOFT']), 'SOFT', np.where(x.shift(-1).Compound.isin(['MEDIUM']), 'MEDIUM', np.where(x.shift(-1).Compound.isin(['HARD']), 'HARD', None))))
    # rename the compound column to strategy before the pit stop
    .rename(columns={'Compound': 'strategy_before_pit'})
    # filter only for laps with pit stops
    .query('pit_stop.notnull()')
    [
['Driver', 'DriverNumber', 'LapNumber', 'PitInTime', 'FreshTyre', 'TyreLife', 'TrackStatus', 'pit_duration', 'pit_stop', 'strategy_after_pit', 'strategy_before_pit']
    ]
)

laps

,Driver,DriverNumber,LapNumber,PitInTime,FreshTyre,TyreLife,TrackStatus,pit_duration,pit_stop,strategy_after_pit,strategy_before_pit
33,VER,33,34.0,0 days 01:16:10.012000,False,39.0,1,23.459,34,HARD,SOFT
109,SAI,55,32.0,0 days 01:13:47.767000,False,38.0,1,23.893,32,HARD,SOFT
185,NOR,4,30.0,0 days 01:11:21.265000,False,35.0,1,22.864,30,HARD,SOFT
268,PER,11,35.0,0 days 01:17:43.911000,False,40.0,1,24.298,35,HARD,SOFT
342,VET,5,31.0,0 days 01:12:42.271000,False,36.0,1,24.348,31,HARD,SOFT
419,GAS,10,30.0,0 days 01:11:23.122000,False,36.0,1,24.165,30,HARD,SOFT
496,HAM,44,29.0,0 days 01:10:07.703000,False,33.0,1,23.663,29,HARD,SOFT
534,HAM,44,67.0,0 days 01:58:47.992000,True,38.0,1,24.006,67,SOFT,HARD
603,STR,18,58.0,0 days 01:47:28.637000,True,58.0,1,23.776,58,SOFT,HARD
659,OCO,31,37.0,0 days 01:20:48.031000,True,37.0,1,23.699,37,MEDIUM,SOFT


In [63]:
# get the number of pit stops made for each driver
pit_stop_counts = laps.groupby('Driver')['pit_stop'].nunique().reset_index().rename(columns={'pit_stop': 'num_pit_stops'})
pit_stop_counts


laps_pit_stops = pd.merge(left=laps, right=pit_stop_counts, on='Driver', how='left')
laps_pit_stops

,Driver,DriverNumber,LapNumber,PitInTime,FreshTyre,TyreLife,TrackStatus,pit_duration,pit_stop,strategy_after_pit,strategy_before_pit,num_pit_stops
0,VER,33,34.0,0 days 01:16:10.012000,False,39.0,1,23.459,34,HARD,SOFT,1
1,SAI,55,32.0,0 days 01:13:47.767000,False,38.0,1,23.893,32,HARD,SOFT,1
2,NOR,4,30.0,0 days 01:11:21.265000,False,35.0,1,22.864,30,HARD,SOFT,1
3,PER,11,35.0,0 days 01:17:43.911000,False,40.0,1,24.298,35,HARD,SOFT,1
4,VET,5,31.0,0 days 01:12:42.271000,False,36.0,1,24.348,31,HARD,SOFT,1
5,GAS,10,30.0,0 days 01:11:23.122000,False,36.0,1,24.165,30,HARD,SOFT,1
6,HAM,44,29.0,0 days 01:10:07.703000,False,33.0,1,23.663,29,HARD,SOFT,2
7,HAM,44,67.0,0 days 01:58:47.992000,True,38.0,1,24.006,67,SOFT,HARD,2
8,STR,18,58.0,0 days 01:47:28.637000,True,58.0,1,23.776,58,SOFT,HARD,1
9,OCO,31,37.0,0 days 01:20:48.031000,True,37.0,1,23.699,37,MEDIUM,SOFT,1


In [ ]:
# Flatten/pivot the above such that we have one row per driver, and columns for each pit stop (lap number, duration, strategy before and after the pit stop)
laps_pit_stops['stop_num'] = laps_pit_stops.groupby('Driver').cumcount() + 1
per_stop_cols = {
        "LapNumber": "lap",
        "pit_duration": "duration",
        "PitInTime": "intime",
        "FreshTyre": "fresh",
        "strategy_before_pit": "tyre_before",
        "strategy_after_pit": "tyre_after",
    }

pivoted = laps_pit_stops.groupby('Driver').first()[["DriverNumber", "num_pit_stops"]].reset_index()


for src_col, label in per_stop_cols.items():
    stop_pivot = (
            laps_pit_stops.pivot_table(index='Driver', columns="stop_num", values=src_col, aggfunc="first")
        )
    stop_pivot.columns = [f"pit_{int(n)}_{label}" for n in stop_pivot.columns]
    pivoted = pivoted.merge(stop_pivot.reset_index(), on='Driver', how="left")


pivoted.sort_values('DriverNumber').reset_index(drop=True)


,Driver,DriverNumber,num_pit_stops,pit_1_lap,pit_2_lap,pit_1_duration,pit_2_duration,pit_1_intime,pit_2_intime,pit_1_fresh,pit_2_fresh,pit_1_tyre_before,pit_2_tyre_before,pit_1_tyre_after,pit_2_tyre_after
0,GAS,10,1,30.0,NaN,24.165,NaN,0 days 01:11:23.122000,NaT,False,NaN,SOFT,NaN,HARD,NaN
1,PER,11,1,35.0,NaN,24.298,NaN,0 days 01:17:43.911000,NaT,False,NaN,SOFT,NaN,HARD,NaN
2,ALO,14,1,45.0,NaN,23.574,NaN,0 days 01:31:34.693000,NaT,True,NaN,MEDIUM,NaN,SOFT,NaN
3,STR,18,1,58.0,NaN,23.776,NaN,0 days 01:47:28.637000,NaT,True,NaN,HARD,NaN,SOFT,NaN
4,TSU,22,1,64.0,NaN,24.226,NaN,0 days 01:56:09.006000,NaT,True,NaN,HARD,NaN,SOFT,NaN
5,RIC,3,1,36.0,NaN,24.613,NaN,0 days 01:19:52.275000,NaT,True,NaN,MEDIUM,NaN,HARD,NaN
6,OCO,31,1,37.0,NaN,23.699,NaN,0 days 01:20:48.031000,NaT,True,NaN,SOFT,NaN,MEDIUM,NaN
7,VER,33,1,34.0,NaN,23.459,NaN,0 days 01:16:10.012000,NaT,False,NaN,SOFT,NaN,HARD,NaN
8,NOR,4,1,30.0,NaN,22.864,NaN,0 days 01:11:21.265000,NaT,False,NaN,SOFT,NaN,HARD,NaN
9,HAM,44,2,29.0,67.0,23.663,24.006,0 days 01:10:07.703000,0 days 01:58:47.992000,False,True,SOFT,HARD,HARD,SOFT
